# Module `dynamic_network.py`

Ce notebook explique le simulateur dynamique du reseau. Il fait evoluer les vraies aretes du graphe residuel d'un tour sur l'autre, tout en garantissant que le reseau reste exploitable.

`DynamicNetworkSimulator` est volontairement decouple du generateur. Le generateur construit une instance statique valide ; le simulateur prend cette instance comme monde de depart et produit une suite de `DynamicGraphSnapshot`.

La dynamique combine deux phenomenes :

- les couts des aretes actives varient ;
- certaines aretes peuvent devenir temporairement indisponibles, puis revenir.

Apres chaque changement, la fermeture metrique est recalculee. Les solveurs ne voient donc jamais un reseau casse ou incomplet : ils recoivent une matrice complete correspondant au tour courant.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.dynamic_network import DynamicNetworkSimulator, active_edge_costs_from_availability
from cesipath.graph_generator import GraphGenerator
from cesipath.models import GraphGenerationConfig

## 1. Ce que fait le simulateur

A chaque tour :

1. Les aretes actives peuvent passer OFF selon `dynamic_forbid_probability`.
2. Les aretes OFF peuvent revenir ON selon `dynamic_restore_probability`.
3. Chaque coupure candidate est verifiee par `DynamicStateValidator` avant d'etre acceptee.
4. Les couts des aretes actives sont re-echantillonnes via `sample_dynamic_edge_cost`.
5. La matrice residuelle et la fermeture metrique sont reconstruites.

Le simulateur ne modifie pas `GraphInstance`. Il copie les dictionnaires du snapshot precedent, applique les changements, puis renvoie un nouveau snapshot. Cela rend l'enchainement des tours facile a suivre dans les notebooks : `snapshot = simulator.advance(snapshot)`.

**Idee cle** : la dynamique agit sur le reseau physique, pas directement sur les tournees. Les algorithmes de resolution ne sont relances que plus haut dans la pile, via `SolverInput` construit a partir du snapshot courant.


## 2. Creer un simulateur et initialiser

`initialize_snapshot()` demarre tous les couts dynamiques au niveau du cout statique, avec toutes les aretes residuelles utilisables sur ON.

Ce point de depart est volontairement neutre : au tour 0, la dynamique n'a encore rien perturbe. La matrice complete du snapshot initial doit donc correspondre a la fermeture metrique du graphe residuel statique.

Le simulateur recoit sa propre seed. C'est important : deux simulateurs construits sur la meme instance peuvent produire deux scenarios dynamiques differents, sans regenerer le graphe de base.


In [ ]:
config = GraphGenerationConfig(node_count=10, seed=42,
                              dynamic_sigma=0.18,
                              dynamic_forbid_probability=0.08,
                              dynamic_restore_probability=0.2)
instance = GraphGenerator(config).generate()

simulator = DynamicNetworkSimulator(instance, seed=7)
snapshot = simulator.initialize_snapshot()

print("Tour           :", snapshot.step)
print("Aretes totales :", len(snapshot.edge_availability))
print("Aretes actives :", snapshot.active_edge_count)
print("Premier cout   :", next(iter(snapshot.edge_costs.items())))
print("Premier chemin :", next(iter(snapshot.completed_paths.items())))

## 3. Logique d'`advance()` en detail

Ordre exact des operations pour chaque arete non `FORBIDDEN` :

```text
si arete ON :
    tirer 'forbid' ~ Uniform(0, 1)
    si forbid < dynamic_forbid_probability ET can_disable_edge(...) :
        passer OFF
sinon (arete OFF) :
    tirer 'restore' ~ Uniform(0, 1)
    si restore < dynamic_restore_probability :
        repasser ON

si arete ON apres cette decision :
    re-echantillonner son cout dynamique
```

Deux details comptent.

D'abord, une arete interdite statiquement est ignoree : elle n'est jamais candidate a la dynamique, car elle ne fait pas partie du reseau exploitable.

Ensuite, on teste la coupure **avant** de l'appliquer. `can_disable_edge` simule le retrait de l'arete candidate et refuse l'action si elle casserait les invariants. La dynamique est donc aleatoire dans ses intentions, mais contrainte dans ses effets.


## 4. La validation refuse certaines coupures

Pour observer le filet de securite, on boucle sur plusieurs tours en trackant le nombre d'aretes actives.

Un tirage peut vouloir couper une arete, mais cette coupure n'est acceptee que si le graphe restant respecte encore : connexite, densite minimale, degre moyen minimal et ratio maximal d'aretes OFF. C'est ce qui empeche une simulation de devenir inutilisable apres quelques tours.


In [ ]:
simulator = DynamicNetworkSimulator(instance, seed=7)
snapshot = simulator.initialize_snapshot()

print(f"tour={snapshot.step:>2}  actives={snapshot.active_edge_count:>3}")
for _ in range(20):
    snapshot = simulator.advance(snapshot)
    print(f"tour={snapshot.step:>2}  actives={snapshot.active_edge_count:>3}")

Meme avec `dynamic_forbid_probability` eleve, le nombre d'aretes actives ne descend jamais sous le plancher fixe par :

- la connexite, qui impose au minimum `n-1` aretes dans un graphe connecte ;
- `resolved_dynamic_min_density` ;
- `resolved_dynamic_min_average_degree` ;
- `dynamic_max_disabled_ratio`.

Ces contraintes se recoupent mais ne sont pas redondantes. Une densite suffisante ne garantit pas la connexite ; une connexite seule peut donner une chaine fragile, propriete classique des graphes non orientes [2] ; le ratio OFF limite les scenarios ou presque tout le reseau dynamique serait coupe en meme temps.


## 5. Recalcul Dijkstra a chaque tour

Apres chaque `advance`, la methode privee `_build_snapshot` :

1. filtre les aretes actives avec `active_edge_costs_from_availability` ;
2. construit `residual_costs` avec `build_cost_matrix` ;
3. applique `complete_graph_with_shortest_paths` pour obtenir `completed_costs` et `completed_paths` via Dijkstra [1] ;
4. emballe le tout dans un `DynamicGraphSnapshot`.

**Pourquoi recalculer toute la fermeture metrique ?** Une seule arete qui passe OFF peut changer le plus court chemin de nombreuses paires de sommets. Inversement, une hausse de cout locale peut rendre un detour preferable. Recalculer depuis les aretes actives garantit que la matrice donnee au solveur represente exactement le reseau du tour courant.

Pour les tailles du projet, cette approche simple est suffisante et beaucoup plus sure qu'un cache partiel. Si les graphes devenaient tres grands, ce module serait l'endroit naturel pour introduire une version incrementale, famille d'algorithmes etudiee pour les plus courts chemins dynamiques [3].


In [ ]:
print("Ligne 0 de la matrice complete :")
print(snapshot.completed_costs[0])
print("\nChemin (0, 3) :", snapshot.completed_paths.get((0, 3)))

## 6. Helper : `active_edge_costs_from_availability`

Cette fonction libre filtre un dict de couts en ne gardant que les aretes marquees ON.

Elle est exposee parce qu'elle est utile en debug et hors du simulateur. Par exemple, on peut inspecter le sous-graphe actif d'un snapshot, recalculer une matrice a la main ou verifier qu'une strategie d'execution n'emprunte pas une arete temporairement indisponible.

Le helper ne valide rien : il applique simplement le masque `edge_availability`. La responsabilite des invariants reste dans `DynamicStateValidator`.


In [ ]:
subset = active_edge_costs_from_availability(snapshot.edge_costs, snapshot.edge_availability)
print("Nb total d'aretes connues   :", len(snapshot.edge_costs))
print("Nb d'aretes actives filtrees :", len(subset))

## 7. Invariants finaux

Apres un appel a `advance()`, les proprietes suivantes sont vraies par construction :

- le graphe actif est connexe ;
- la densite active reste au-dessus du seuil dynamique ;
- le degre moyen actif reste suffisant ;
- le ratio d'aretes OFF reste borne ;
- la matrice `completed_costs` est une fermeture metrique du graphe actif ;
- chaque `completed_paths[(i, j)]` utilise uniquement des aretes actives.

Ces invariants sont ce qui rend la comparaison des strategies dynamique possible. Quand une re-optimisation donne un meilleur cout qu'un plan fige, on veut que la difference vienne de la strategie, pas d'un bug ou d'un reseau devenu invalide.

A retenir : `dynamic_network.py` est le chef d'orchestre de la simulation. Il ne decide pas comment optimiser une tournee, mais il garantit que chaque tour fournit un probleme VRP coherent.

---

## References

[1] **Cormen, T. H., Leiserson, C. E., Rivest, R. L. & Stein, C. (2022).** *Introduction to Algorithms* (4th ed.). MIT Press.

[2] **Bondy, J. A. & Murty, U. S. R. (2008).** *Graph Theory.* Springer. https://doi.org/10.1007/978-1-84628-970-5

[3] **Ramalingam, G. & Reps, T. (1996).** *An incremental algorithm for a generalization of the shortest-path problem.* Journal of Algorithms, 21(2), 267-305. https://doi.org/10.1006/jagm.1996.0046
